[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t08_inverse.ipynb)

# From the height back to the angle: inverse trig, and how your phone knows it's upside down

Your phone feels gravity as three numbers. To get its tilt it needs an angle from a ratio: an inverse tangent. In
NXP's own worked example, the plain inverse tangent gives −5.29°, but the phone is lying face down, at 174.71°. The
fix, called atan2, reads the signs.

The idea: **many turns give the same height. To go backwards, we have to choose one.**

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · One height, many turns

Pick a height for the arrow's tip, between −1 and 1. Two arrows in every full turn have that height: one on the
right, one on the left. Keep turning and there are infinitely many. Going backwards from a height to an angle, we
must name one of them. That chosen one is the **principal value**.

In [ ]:
def one_shadow(s=0.5):
    p = math.asin(s)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(12.5, 5), gridspec_kw={"width_ratios": [1, 1.9]})
    circle_axes(ax)
    ax.axhline(s, color=SAGE, lw=1.4, ls="--")
    arrow(ax, math.pi - p, color=TEAL, lw=2.4, label=f"{math.degrees(math.pi - p):.0f}°")
    arrow(ax, p, label=f"{math.degrees(p):.0f}° (principal)")
    ax.set_title(f"height (vertical projection) = {s:+.2f}")
    x = np.linspace(-2 * np.pi, 3 * np.pi, 1500)
    bx.plot(x, np.sin(x), color=DIM, lw=1.4)
    seg = np.linspace(-np.pi / 2, np.pi / 2, 300)
    bx.plot(seg, np.sin(seg), color=CORAL, lw=4, label="the principal branch, −π/2 to π/2")
    bx.axhline(s, color=SAGE, lw=1.4, ls="--")
    sols = sorted([p + 2 * np.pi * k for k in range(-1, 2)] + [np.pi - p + 2 * np.pi * k for k in range(-1, 2)])
    for v in sols:
        bx.plot([v], [s], "o", color=CORAL if abs(v - p) < 1e-9 else TEAL, ms=9)
    bx.set_ylim(-1.4, 1.6); pi_ticks(bx, -2 * np.pi, 3 * np.pi); bx.legend(loc="upper left")
    bx.set_title(f"every angle with sine {s:+.2f}: {len(sols)} shown, infinitely many in all")
    plt.tight_layout(); plt.show()
    print(f"sin⁻¹({s:+.2f}) = {p:+.4f} rad = {math.degrees(p):+.1f}°   (the principal value)")

interact(one_shadow, s=FloatSlider(value=0.5, min=-1, max=1, step=0.05, description="height"));

**Notice.** The coral piece of the sine wave, from −π/2 to π/2, rises steadily and passes every height from −1 to 1
exactly once. That is why it is the chosen branch for sin⁻¹: on it, every height has exactly one angle.

## 2 · The six principal branches

Each inverse function uses one piece of its trig graph that rises or falls steadily, and passes every value once.
Turn that piece on its side (reflect it in the line y = x) and you have the inverse graph.

In [ ]:
INV = {
    "sin⁻¹": (np.sin, np.arcsin, (-np.pi / 2, np.pi / 2), "[−1, 1]", "[−π/2, π/2]"),
    "cos⁻¹": (np.cos, np.arccos, (0, np.pi), "[−1, 1]", "[0, π]"),
    "tan⁻¹": (np.tan, np.arctan, (-np.pi / 2 + 0.05, np.pi / 2 - 0.05), "all real numbers", "(−π/2, π/2)"),
    "cot⁻¹": (lambda v: 1 / np.tan(v), lambda v: np.pi / 2 - np.arctan(v), (0.05, np.pi - 0.05), "all real numbers", "(0, π)"),
    "sec⁻¹": (lambda v: 1 / np.cos(v), lambda v: np.arccos(1 / v), (0, np.pi), "x ≤ −1 or x ≥ 1", "[0, π] except π/2"),
    "cosec⁻¹": (lambda v: 1 / np.sin(v), lambda v: np.arcsin(1 / v), (-np.pi / 2, np.pi / 2), "x ≤ −1 or x ≥ 1", "[−π/2, π/2] except 0"),
}
print(f"{'function':9} {'domain':20} {'range (principal value branch)':30}")
for k, (_, _, _, dom, rng_) in INV.items(): print(f"{k:9} {dom:20} {rng_:30}")

def branch(name="sin⁻¹"):
    f, finv, (a, b), dom, rng_ = INV[name]
    t = np.linspace(a, b, 800)
    with np.errstate(divide="ignore", invalid="ignore"):
        y = f(t)
    keep = np.abs(y) < 4
    fig, ax = plt.subplots(figsize=(6.8, 6.8))
    ax.plot([-4, 4], [-4, 4], color=DIM, ls="--", lw=1, label="y = x (the mirror)")
    ax.plot(np.where(keep, t, np.nan), np.where(keep, y, np.nan), color=TEAL, lw=2.4, label=f"{name.rstrip('⁻¹')}, on its branch")
    ax.plot(np.where(keep, y, np.nan), np.where(keep, t, np.nan), color=CORAL, lw=3.2, label=name)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect("equal"); ax.legend(loc="lower right")
    ax.set_title(f"{name}: domain {dom}; range {rng_}", fontsize=12)
    plt.show()

interact(branch, name=Dropdown(options=list(INV), value="sin⁻¹", description="function"));

**Notice.** The coral inverse graph is the teal branch flipped over the dashed mirror. Note that sin⁻¹ x is not
1 ÷ sin x: the little ⁻¹ means "the inverse function" (John Herschel's notation, 1813).

## 3 · There and back: sin(sin⁻¹ x) and sin⁻¹(sin x)

Go from a height to an angle and back, and you always return: **sin(sin⁻¹ x) = x** for every x in [−1, 1].

Go from an angle to a height and back, and you return only if you started on the principal branch:
**sin⁻¹(sin x) = x** only for x in [−π/2, π/2]. Outside it, you land on the branch's angle with the same height. The
graph becomes a zigzag.

In [ ]:
def there_and_back(x_deg=150.0):
    x0 = math.radians(x_deg)
    x = np.linspace(-2 * np.pi, 2 * np.pi, 2000)
    fig, ax = plt.subplots(figsize=(11, 4.6))
    ax.plot(x, x, color=DIM, ls="--", lw=1, label="y = x")
    ax.plot(x, np.arcsin(np.sin(x)), color=CORAL, lw=2.6, label="sin⁻¹(sin x)")
    ax.axvspan(-np.pi / 2, np.pi / 2, color=SAGE, alpha=0.12, label="the principal branch")
    ax.plot([x0], [math.asin(math.sin(x0))], "o", color=CHALK, ms=10)
    ax.set_ylim(-2, 2); pi_ticks(ax); ax.legend(loc="upper left", fontsize=11)
    ax.set_title(f"sin⁻¹(sin {x_deg:.0f}°) = {math.degrees(math.asin(math.sin(x0))):.0f}°")
    plt.show()

interact(there_and_back, x_deg=FloatSlider(value=150, min=-360, max=360, step=5, description="x (°)"));

**Notice.** sin⁻¹(sin 150°) is 30°, not 150°: both angles give the same height, and 30° is the one on the branch.

## 4 · Your phone, face down

A phone's accelerometer measures gravity along its three edges. NXP's application note AN3461 works an example with
readings Gx = 0.461105, Gy = 0.082198, Gz = −0.887432 (in units of g). The roll angle φ satisfies
tan φ = Gy ÷ Gz.

tan⁻¹ can only answer between −90° and 90°. A phone lying face down is outside that range, so tan⁻¹ gives the wrong
one of the two angles with this tangent. **atan2(Gy, Gz)** looks at the signs of both numbers and picks the right
quarter of the circle.

In [ ]:
Gx, Gy, Gz = 0.461105, 0.082198, -0.887432
roll_atan = math.degrees(math.atan(Gy / Gz))
roll_atan2 = math.degrees(math.atan2(Gy, Gz))
pitch = math.degrees(math.atan(-Gx / math.hypot(Gy, Gz)))
print(f"tan φ = {Gy / Gz:.6f}")
print(f"tan⁻¹ says  φ = {roll_atan:.2f}°")
print(f"atan2 says  φ = {roll_atan2:.2f}°   (the phone is face down)")
print(f"pitch = {pitch:.4f}°  (this one is fine with tan⁻¹: pitch only runs from −90° to 90°)")
assert (round(roll_atan, 2), round(roll_atan2, 2)) == (-5.29, 174.71)          # NXP rounds tan φ first, so their 4th decimal differs

In [ ]:
def tilt(roll_deg=170.0):
    r = math.radians(roll_deg)
    gy, gz = math.sin(r), math.cos(r)                    # gravity seen by a phone rolled by `roll_deg`
    a1, a2 = math.atan(gy / gz) if abs(gz) > 1e-9 else math.copysign(math.pi / 2, gy), math.atan2(gy, gz)
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    circle_axes(ax)
    arrow(ax, a2, color=TEAL, lw=4, label=f"atan2: {math.degrees(a2):.1f}°")
    arrow(ax, a1, label=f"tan⁻¹: {math.degrees(a1):.1f}°")
    ax.axvspan(0, 1.35, color=SAGE, alpha=0.08)
    ax.set_title(f"true roll {roll_deg:.0f}°  (shaded: where tan⁻¹ can answer)", fontsize=12)
    plt.show()

interact(tilt, roll_deg=FloatSlider(value=170, min=-179, max=179, step=1, description="roll (°)"));

**Notice.** While the roll stays between −90° and 90°, the two arrows agree. Roll past 90° and tan⁻¹ jumps to the
other side of the circle: same tangent, wrong angle.

## 5 · Every solution of sin θ = sin α (class 11, assessed formatively only)

CBSE includes this for formative assessment only. Every angle with the same height (sine) as α is
**θ = nπ + (−1)ⁿ α**, for any whole number n. For the same cosine, **θ = 2nπ ± α**; for tan, **θ = nπ + α**.

In [ ]:
alpha = math.radians(30)
sols = [n * math.pi + (-1) ** n * alpha for n in range(-3, 4)]
print("θ = nπ + (−1)ⁿ α with α = 30°:", [f"{math.degrees(v):.0f}°" for v in sols])
assert all(abs(math.sin(v) - math.sin(alpha)) < 1e-12 for v in sols)

## 6 · The identity gallery

**sin(sin⁻¹ x) = x** on [−1, 1], and **sin⁻¹(sin x) = x** on [−π/2, π/2]: the two properties the syllabus keeps. The
same pattern for cos and tan, each on its own branch.

In [ ]:
xs = np.linspace(-1, 1, 400)
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4))
both_sides(axs[0], xs, np.sin(np.arcsin(xs)), xs, "sin(sin⁻¹ x)  and  x,  on [−1, 1]")
axs[0].set_xticks([-1, -0.5, 0, 0.5, 1]); axs[0].set_xticklabels(["−1", "−0.5", "0", "0.5", "1"])
xb = np.linspace(-np.pi / 2, np.pi / 2, 400)
both_sides(axs[1], xb, np.arcsin(np.sin(xb)), xb, "sin⁻¹(sin x)  and  x,  on [−π/2, π/2]")
axs[0].legend(loc="lower right"); plt.tight_layout(); plt.show()
check_identity(lambda x: np.sin(np.arcsin(x)), lambda x: x, lo=-1, hi=1)
check_identity(lambda x: np.arcsin(np.sin(x)), lambda x: x, lo=-np.pi / 2, hi=np.pi / 2)
check_identity(lambda x: np.cos(np.arccos(x)), lambda x: x, lo=-1, hi=1)
check_identity(lambda x: np.arccos(np.cos(x)), lambda x: x, lo=0, hi=np.pi)
check_identity(lambda x: np.tan(np.arctan(x)), lambda x: x, lo=-50, hi=50)
check_identity(lambda x: np.arctan(np.tan(x)), lambda x: x, lo=-np.pi / 2 + 0.01, hi=np.pi / 2 - 0.01);

**The branches' ranges, checked.** Every inverse function's answers land inside its principal branch.

In [ ]:
rng = np.random.default_rng(1)
for name, (_, finv, _, dom, rng_) in INV.items():
    v = rng.uniform(-1, 1, 2000) if name in ("sin⁻¹", "cos⁻¹") else rng.uniform(-50, 50, 2000)
    if name in ("sec⁻¹", "cosec⁻¹"): v = v[np.abs(v) >= 1]
    out = finv(v)
    print(f"{name:8} answers from {out.min():+.4f} to {out.max():+.4f}   (branch: {rng_})")

## 7 · On paper: the CBSE way

**Example 1.** Find the principal value of sin⁻¹(1/√2).

1. We want the angle in [−π/2, π/2] whose sine is 1/√2.
2. sin(π/4) = 1/√2, and π/4 is on the branch.
3. **sin⁻¹(1/√2) = π/4.**

In [ ]:
print(f"sin⁻¹(1/√2) = {math.asin(1 / math.sqrt(2)):.6f},  π/4 = {math.pi / 4:.6f}")

**Example 2.** Find the principal value of cos⁻¹(−1/2).

1. We want the angle in [0, π] whose cosine is −1/2.
2. cos(π/3) = 1/2, and the horizontal projection points left in the second quarter: cos(π − π/3) = −1/2.
3. **cos⁻¹(−1/2) = 2π/3.**

In [ ]:
print(f"cos⁻¹(−1/2) = {math.acos(-0.5):.6f},  2π/3 = {2 * math.pi / 3:.6f}")

**Try it.**
1. Find the principal value of cosec⁻¹(2).
2. Find tan⁻¹(1) + cos⁻¹(−1/2) + sin⁻¹(−1/2).

<details><summary>Answers</summary>

1. cosec⁻¹(2) = sin⁻¹(1/2) = π/6.
2. π/4 + 2π/3 − π/6 = 3π/4.

</details>

In [ ]:
assert abs(math.atan(1) + math.acos(-0.5) + math.asin(-0.5) - 3 * math.pi / 4) < 1e-12
print("try-it 2 checked")

## Check yourself
1. What is sin⁻¹(sin 150°)?
2. What is the range of cos⁻¹?
3. Why does the plain inverse tangent get a face-down phone wrong?

<details><summary>Answers</summary>

1. 30°: the angle on the branch with the same height.
2. [0, π].
3. Its answers stay between −90° and 90°, but a face-down phone is rolled beyond 90°. atan2 uses both signs to pick
   the right quarter.

</details>

## Sources
NXP (Freescale) application note AN3461, "Tilt Sensing Using a Three-Axis Accelerometer", M. Pedley, rev. 6, 2013.
NCERT class 12, chapter 2 (principal value branches). CBSE Mathematics syllabus 2026–27 (class 12 inverse
trigonometric functions; class 11 general solutions, assessed formatively). All retrieved 2026-10-21.